# LIFE: station keeping and octahedral formation manoeuvres

This notebook studies how an octahedral formation follows a fixed offset, a rotation, and a smooth re-pointing around a reference spacecraft, **Base**, near Sun–Earth L2. It compares correction intervals and moving-reference burn spacings using Astrogator propagation, endpoint checks, tracking-error histories and executable Δv.

The experiment has four stages:

1. Define and animate the desired six-vertex geometry in Python.
2. Create a fresh STK scenario and propagate Base from an embedded Cartesian state.
3. Run fixed-offset station keeping for the selected spacecraft and correction intervals.
4. Follow a rotating reference using two-impulse transfers, compare burn spacings with a matched inertial hold, and optionally run a re-pointing manoeuvre.

A **follower** is one simulated spacecraft near Base. A **waypoint** is its desired position and velocity at a chosen time. The code propagates freely between waypoints; it does not force the spacecraft to follow the reference curve continuously. All six reference vertices are always drawn, but only the selected vertices receive dynamical simulations and Δv calculations.

Run from the top in a Windows Jupyter kernel with **STK 13 Desktop, an Astrogator licence, the `agi.stk13` Python API, NumPy, SciPy, Pandas and Matplotlib**. The geometry cells run without STK. Every function and numerical input is defined in this notebook. No previous notebook, open scenario, helper module, report style, JWST ephemeris, or network download is needed. CSVs, figures, HTML animations and a scenario checkpoint are written to a new results folder.

The numerical settings are development choices, not established LIFE mission requirements. This is an impulsive reference-tracking experiment, not a minimum-fuel controller or a validated operational formation design.


## 1. Imports and settings

These are the settings to edit before running the notebook. Restart the kernel and run from the top after changing them.

| Setting | Meaning |
|---|---|
| `NUM_SPACECRAFT` | Number of simulated followers, from 1 to 6; Base is additional. |
| `SPACECRAFT_LABELS` | `None` selects the first N labels in `Xp, Xm, Yp, Ym, Zp, Zm`. For a particular subset, use e.g. `['Xp', 'Zp']` with N = 2. |
| `SK_INTERVALS_DAYS` | Separate position-only fixed-offset baseline experiments, e.g. `[7, 14]`. |
| `BURN_SPACINGS_HOURS` | Separate moving-reference experiments, e.g. `[3, 6]`. Each value is the time between successive waypoints and executable burns. |
| `ROTATION_PERIOD_DAYS` | Time for one complete 360° reference rotation. Increasing it slows the rotation. |
| `TRACK_DURATION_DAYS` | Length actually simulated for rotation and matched hold. `None` runs one full rotation; `1.0` runs its first day. |
| `BASELINE_DURATION_DAYS` | Common duration for all fixed-offset baseline cases. |
| `L_M`, `LOS_START`, `LOS_END` | Centre-to-vertex distance and initial/final line-of-sight directions in ICRF axes. |
| `START_OFFSET_DAYS` | Delay along the newly propagated Base trajectory before releasing followers; this is a first halo-phase sensitivity setting. |
| `SAMPLE_S` | Reporting cadence; it changes error sampling, not burn timing or the numerical integrator. |
| `POSITION_TOL_M` | Numerical endpoint tolerance per coordinate, not a continuous tracking requirement. |

The two interval lists describe **different experiments**. They are not multiplied together: a 7-day baseline and a 3-hour rotation are separate runs. To study rotation with 7-day burns, explicitly put `168` in `BURN_SPACINGS_HOURS`.

Suggested first changes are spacecraft subset, burn spacing, size and rotation period, one at a time. Start with a one-day dynamical arc. A full 28-day rotation of six spacecraft at 3-hour spacing requires 1,344 targeted rotation legs, before the matched holds and other cases. Smaller spacing usually reduces interpolation of the curved reference by coast arcs, but can change Δv and increase runtime.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from functools import lru_cache
from itertools import combinations
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from scipy.spatial.transform import Rotation
from scipy.integrate import trapezoid

NUM_SPACECRAFT = 1
SPACECRAFT_LABELS = None               # Example: ['Xp', 'Zp'] with NUM_SPACECRAFT = 2.
SK_INTERVALS_DAYS = [7.0, 14.0]
BURN_SPACINGS_HOURS = [3.0, 6.0]
ROTATION_PERIOD_DAYS = 28.0
TRACK_DURATION_DAYS = 1.0              # None = one complete rotation.
BASELINE_DURATION_DAYS = 21.0

L_M = 1000.0                           # Centre-to-vertex distance, not edge length.
LOS_START = np.array([0.0, 0.0, 1.0])
LOS_END = np.array([np.sin(np.deg2rad(20)), 0.0, np.cos(np.deg2rad(20))])
SLEW_DURATION_DAYS = 2.0
START_OFFSET_DAYS = 0.0
SAMPLE_S = 600.0
POSITION_TOL_M = 0.10
BASELINE_POSITION_TOL_M = 0.01          # Original fixed-offset target tolerance.
VELOCITY_TOL_MPS = 1e-6
DC_PERTURBATION_MPS = 1e-4
DC_MAX_STEP_MPS = 0.1
DC_MAX_ITERATIONS = 30

RUN_BASELINE = True
RUN_SINGLE_TRANSFER = True
RUN_ROTATION = True
RUN_MATCHED_HOLD = True
RUN_REPOINTING = False
STOP_AT_END = True                    # Rotation/hold include stopping relative to Base.
ANIMATE_REFERENCES = True
ANIMATE_ACTUAL = True
ANIMATION_FRAMES = 101
ANIMATION_INTERVAL_MS = 70
SAVE_SCENARIO = True

PROPAGATOR_NAME = 'Earth Default High Fidelity v13'
BASE_SEED_UTCG = '6 Oct 2022 07:10:00.000'
# Earth-centred ICRF, metres and m/s; provenance is explained in section 10.
BASE_SEED_R_M = np.array([1.15803168e9, 9.39831792e7, 4.32308211e8])
BASE_SEED_V_MPS = np.array([-124.13911218, 510.44906145, 237.63841329])
DRY_MASS_KG = 50.0
FUEL_MASS_KG = 50.0
AREA_M2 = 2.0
CR = 1.0
CD = 2.2

plt.rcParams.update({'font.size': 11, 'figure.figsize': (10, 5),
                     'animation.embed_limit': 100.0})


## 2. Validate the choices and print the run plan

Convert days and hours into seconds once. The full rotation period and simulated arc remain separate variables. Each case starts at the same epoch and uses the same Base trajectory. A short final interval is included when a duration is not divisible by its burn spacing.

The plan counts targeted coast legs, not targeter iterations. Astrogator reruns previously completed segments when a growing mission sequence is executed, so runtime can increase faster than this count. Results are saved after each successful case.


In [ ]:
LABELS = ['Xp', 'Xm', 'Yp', 'Ym', 'Zp', 'Zm']
if not isinstance(NUM_SPACECRAFT, int) or isinstance(NUM_SPACECRAFT, bool) or not 1 <= NUM_SPACECRAFT <= 6:
    raise ValueError('NUM_SPACECRAFT must be an integer from 1 to 6.')
SELECTED_LABELS = LABELS[:NUM_SPACECRAFT] if SPACECRAFT_LABELS is None else list(SPACECRAFT_LABELS)
if (len(SELECTED_LABELS) != NUM_SPACECRAFT or len(set(SELECTED_LABELS)) != NUM_SPACECRAFT
        or any(label not in LABELS for label in SELECTED_LABELS)):
    raise ValueError('Choose NUM_SPACECRAFT different labels from ' + str(LABELS))
SELECTED_VERTICES = [LABELS.index(label) for label in SELECTED_LABELS]

def positive_list(values, name):
    if not values or any(not np.isfinite(v) or v <= 0 for v in values):
        raise ValueError(name + ' must contain positive, finite numbers.')
    return sorted(set(float(v) for v in values))

SK_INTERVALS_DAYS = positive_list(SK_INTERVALS_DAYS, 'SK_INTERVALS_DAYS')
BURN_SPACINGS_HOURS = positive_list(BURN_SPACINGS_HOURS, 'BURN_SPACINGS_HOURS')
for name in ['L_M', 'ROTATION_PERIOD_DAYS', 'BASELINE_DURATION_DAYS', 'SLEW_DURATION_DAYS',
             'SAMPLE_S', 'POSITION_TOL_M', 'BASELINE_POSITION_TOL_M', 'VELOCITY_TOL_MPS',
             'DC_PERTURBATION_MPS', 'DC_MAX_STEP_MPS', 'DRY_MASS_KG']:
    if not np.isfinite(globals()[name]) or globals()[name] <= 0:
        raise ValueError(name + ' must be positive and finite.')
if not np.isfinite(START_OFFSET_DAYS) or START_OFFSET_DAYS < 0:
    raise ValueError('START_OFFSET_DAYS must be finite and non-negative.')
if TRACK_DURATION_DAYS is not None and (not np.isfinite(TRACK_DURATION_DAYS) or TRACK_DURATION_DAYS <= 0):
    raise ValueError('TRACK_DURATION_DAYS must be positive or None.')
if min(BURN_SPACINGS_HOURS)*3600 < 1 or min(SK_INTERVALS_DAYS)*86400 < 1:
    raise ValueError('Burn and correction spacings must be at least one second.')
if ANIMATION_FRAMES < 2 or DC_MAX_ITERATIONS < 1:
    raise ValueError('Choose at least two animation frames and one targeter iteration.')
if not any([RUN_BASELINE, RUN_SINGLE_TRANSFER, RUN_ROTATION, RUN_MATCHED_HOLD, RUN_REPOINTING]):
    raise ValueError('Enable at least one experiment.')
if any(not np.isfinite(v) or v < 0 for v in [FUEL_MASS_KG, AREA_M2, CR, CD]):
    raise ValueError('Mass, area and optical/aerodynamic coefficients must be non-negative.')
if BASE_SEED_R_M.shape != (3,) or BASE_SEED_V_MPS.shape != (3,) or not np.isfinite(np.r_[BASE_SEED_R_M, BASE_SEED_V_MPS]).all():
    raise ValueError('Base requires two finite three-component vectors.')

DAY_S = 86400.0
ROTATION_PERIOD_S = ROTATION_PERIOD_DAYS * DAY_S
TRACK_DURATION_S = (ROTATION_PERIOD_DAYS if TRACK_DURATION_DAYS is None else TRACK_DURATION_DAYS) * DAY_S
BASELINE_DURATION_S = BASELINE_DURATION_DAYS * DAY_S
SLEW_DURATION_S = SLEW_DURATION_DAYS * DAY_S
DEMO_SPACING_S = max(BURN_SPACINGS_HOURS) * 3600.0
plan = []
for operation, enabled, duration, spacings in [
        ('baseline_L2', RUN_BASELINE, BASELINE_DURATION_S, np.array(SK_INTERVALS_DAYS)*DAY_S),
        ('rotation', RUN_ROTATION, TRACK_DURATION_S, np.array(BURN_SPACINGS_HOURS)*3600),
        ('inertial_hold', RUN_MATCHED_HOLD, TRACK_DURATION_S, np.array(BURN_SPACINGS_HOURS)*3600),
        ('repointing', RUN_REPOINTING, SLEW_DURATION_S, np.array(BURN_SPACINGS_HOURS)*3600)]:
    if enabled:
        for spacing in spacings:
            plan.append(dict(operation=operation, duration_days=duration/DAY_S,
                             spacing_hours=spacing/3600, spacecraft=NUM_SPACECRAFT,
                             targeted_legs=NUM_SPACECRAFT*int(np.ceil(duration/spacing))))
if RUN_SINGLE_TRANSFER:
    plan.append(dict(operation='single_transfer', duration_days=DEMO_SPACING_S/DAY_S,
                     spacing_hours=DEMO_SPACING_S/3600, spacecraft=1, targeted_legs=1))
plan = pd.DataFrame(plan)
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT = Path.cwd() / 'results' / ('LIFE_first_steps_' + RUN_TAG)
OUT.mkdir(parents=True, exist_ok=False)
plan.to_csv(OUT/'run_plan.csv', index=False)
display(plan)
print('Simulated followers:', SELECTED_LABELS, '| Base is additional.')
print('Rotation period:', ROTATION_PERIOD_DAYS, 'days; tracked arc:', TRACK_DURATION_S/DAY_S, 'days')
print('Total targeted legs:', int(plan.targeted_legs.sum()))
print('Outputs:', OUT)


## 3. Define the oriented octahedron

Build the six labelled vertices `(±L,0,0)`, `(0,±L,0)`, `(0,0,±L)`. The +Z axis follows `LOS_START`; projected ICRF +X defines the initial roll. All six vertices define the reference even when fewer spacecraft are propagated. The subset does not change the reference geometry.

The edge length is √2 L and opposite vertices are 2 L apart. Antipodal re-pointing directions need an explicit rotation axis and are rejected here.


In [ ]:
VERTICES = L_M * np.array([[1,0,0], [-1,0,0], [0,1,0], [0,-1,0], [0,0,1], [0,0,-1]], float)
PAIRS = list(combinations(range(6), 2))

def unit(vector):
    vector = np.asarray(vector, dtype=float)
    length = np.linalg.norm(vector)
    if length < 1e-14:
        raise ValueError('A direction must be nonzero.')
    return vector / length

def initial_orientation(los):
    z = unit(los)
    seed = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(seed, z)) > 0.95:
        seed = np.array([0.0, 1.0, 0.0])
    x = unit(seed - np.dot(seed, z) * z)
    y = np.cross(z, x)
    return np.column_stack([x, y, z])

Q0 = initial_orientation(LOS_START)
_slew_cross = np.cross(unit(LOS_START), unit(LOS_END))
SLEW_ANGLE = np.arctan2(np.linalg.norm(_slew_cross), np.dot(unit(LOS_START), unit(LOS_END)))
if np.isclose(SLEW_ANGLE, np.pi):
    raise ValueError('Choose an explicit slew axis for a 180-degree re-pointing.')
SLEW_AXIS = unit(_slew_cross) if np.linalg.norm(_slew_cross) > 1e-14 else Q0[:, 0]
print('Centre-to-vertex:', L_M, 'm; edge:', np.sqrt(2)*L_M, 'm; opposite:', 2*L_M, 'm')


## 4. Desired positions and velocities

`reference(t, operation)` returns six positions in metres and velocities in m/s, relative to Base **with ICRF axis directions**. Rigid rotation uses $\dot\rho=\omega\times\rho$. The two Z-axis reference vertices stay on the rotation axis; the other four rotate because this motion is prescribed, not because gravity produces a rigidly rotating octahedron. All followers still experience differential orbital accelerations.

Re-pointing uses the quintic time law $s(u)=10u^3-15u^4+6u^5$. Angular speed and angular acceleration vanish at its endpoints. The shortest rotation transports the formation without an extra twist. These are translational trajectories, separate from spacecraft attitude.

`inertial_hold` keeps the reference directions fixed in ICRF. The later baseline instead holds coordinates fixed in the rotating L2 frame; these are physically different orientation laws.


In [ ]:
def reference(t_s, operation='rotation'):
    t = float(t_s)
    if operation == 'rotation':
        omega = 2*np.pi / ROTATION_PERIOD_S * unit(LOS_START)
        turn = Rotation.from_rotvec(omega*t).as_matrix()
    elif operation == 'repointing':
        u = np.clip(t / SLEW_DURATION_S, 0.0, 1.0)
        s = 10*u**3 - 15*u**4 + 6*u**5
        ds = (30*u**2 - 60*u**3 + 30*u**4) / SLEW_DURATION_S
        turn = Rotation.from_rotvec(SLEW_AXIS*SLEW_ANGLE*s).as_matrix()
        omega = SLEW_AXIS*SLEW_ANGLE*ds
    elif operation == 'inertial_hold':
        turn = np.eye(3)
        omega = np.zeros(3)
    else:
        raise ValueError(operation)
    position_m = VERTICES @ (turn @ Q0).T
    velocity_mps = np.cross(omega, position_m)
    return position_m, velocity_mps


## 5. Verify and plot the reference geometry

Check all 15 pair distances, all six radii, the centroid, analytic velocities against centred finite differences, rotation periodicity, and the slew endpoints. Then export and plot both reference motions. These are geometric checks, not tests of Astrogator propagation.


In [ ]:
def pair_distances(points):
    return np.array([np.linalg.norm(points[i]-points[j]) for i,j in PAIRS])

reference_distances = pair_distances(VERTICES)
checks = []
for operation, duration in [('rotation', ROTATION_PERIOD_S), ('repointing', SLEW_DURATION_S)]:
    for t in np.linspace(0, duration, 41):
        p, v = reference(t, operation)
        np.testing.assert_allclose(np.linalg.norm(p, axis=1), L_M, atol=1e-9)
        np.testing.assert_allclose(pair_distances(p), reference_distances, atol=1e-9)
        np.testing.assert_allclose(p.mean(axis=0), 0, atol=1e-10)
        if 1 < t < duration-1:
            h = 0.25
            fd = (reference(t+h, operation)[0]-reference(t-h, operation)[0])/(2*h)
            np.testing.assert_allclose(fd, v, rtol=1e-6, atol=1e-8)
    checks.append({'operation': operation, 'geometry_and_velocity_checks': 'passed'})
np.testing.assert_allclose(reference(ROTATION_PERIOD_S)[0], reference(0)[0], atol=1e-9)
for t, los in [(0, LOS_START), (SLEW_DURATION_S, LOS_END)]:
    p, v = reference(t, 'repointing')
    np.testing.assert_allclose(unit(p[4]), unit(los), atol=1e-12)
    np.testing.assert_allclose(v, 0, atol=1e-12)
display(pd.DataFrame(checks))
print('Initial vertex speeds [m/s]:', np.linalg.norm(reference(0)[1], axis=1))


### Reference paths and exports

Plot the desired paths and write the six-vertex reference states to CSV. These paths have no dynamical tracking error.


In [ ]:
fig = plt.figure(figsize=(12, 5))
for panel, (operation, duration) in enumerate([('rotation', ROTATION_PERIOD_S), ('repointing', SLEW_DURATION_S)], 1):
    times = np.linspace(0, duration, 181)
    states = [reference(t, operation) for t in times]
    positions = np.array([s[0] for s in states])
    velocities = np.array([s[1] for s in states])
    ax = fig.add_subplot(1, 2, panel, projection='3d')
    rows = []
    for j, label in enumerate(LABELS):
        ax.plot(*positions[:,j,:].T, label=label)
        ax.scatter(*positions[0,j,:], s=18)
        for k, t in enumerate(times):
            rows.append([t, label, *positions[k,j], *velocities[k,j]])
    ax.set(xlabel='ICRF X [m]', ylabel='ICRF Y [m]', zlabel='ICRF Z [m]', title=operation)
    ax.set_box_aspect((1,1,1))
    ax.legend(fontsize=8)
    pd.DataFrame(rows, columns=['elapsed_s','vertex','x_m','y_m','z_m','vx_mps','vy_mps','vz_mps']).to_csv(OUT/f'reference_{operation}.csv', index=False)
fig.tight_layout()
fig.savefig(OUT/'reference_geometry.png', dpi=180, bbox_inches='tight')
plt.show()


## 6. Animate the reference rotation and re-pointing

Display and save two Python animations like the original formation animation. The twelve edges show the complete reference. Filled red points identify the spacecraft selected for propagation; hollow grey points are reference-only vertices. Base is at the origin. An animation always spans the complete reference operation, even when the later dynamical experiment covers only one day.

The HTML animations include their frames and need no video encoder or external JavaScript file.


In [ ]:
EDGES = [(i,j) for i,j in PAIRS
         if np.isclose(np.linalg.norm(VERTICES[i]-VERTICES[j]), np.sqrt(2)*L_M)]

def show_animation(animation, figure, filename):
    html = animation.to_jshtml()
    (OUT/filename).write_text(html, encoding='utf-8')
    plt.close(figure)
    display(HTML(html))

def animate_reference(operation):
    duration = ROTATION_PERIOD_S if operation == 'rotation' else SLEW_DURATION_S
    times = np.linspace(0, duration, ANIMATION_FRAMES)
    fig = plt.figure(figsize=(7,6))
    ax = fig.add_subplot(projection='3d')
    limit = 1.2*L_M
    ax.set(xlim=(-limit,limit), ylim=(-limit,limit), zlim=(-limit,limit),
           xlabel='ICRF X [m]', ylabel='ICRF Y [m]', zlabel='ICRF Z [m]')
    ax.set_box_aspect((1,1,1))
    lines = [ax.plot([], [], [], color='steelblue', alpha=.65)[0] for _ in EDGES]
    points = ax.scatter([], [], [], facecolors='none', edgecolors='grey', label='Reference vertices')
    selected = ax.scatter([], [], [], color='darkred', label='Selected followers')
    ax.scatter(0,0,0,color='black',marker='+',label='Base')
    texts = [ax.text(0,0,0,label) for label in LABELS]
    ax.legend(fontsize=8)

    def draw_frame(k):
        p, _ = reference(times[k], operation)
        for line, (i,j) in zip(lines, EDGES):
            line.set_data_3d(*p[[i,j]].T)
        points._offsets3d = tuple(p.T)
        selected._offsets3d = tuple(p[SELECTED_VERTICES].T)
        for text, xyz in zip(texts, p):
            text.set_position((xyz[0],xyz[1]))
            text.set_3d_properties(xyz[2], zdir='z')
        ax.set_title(f'{operation}: {times[k]/DAY_S:.2f} days')
        return lines

    animation = FuncAnimation(fig, draw_frame, frames=len(times), interval=ANIMATION_INTERVAL_MS)
    show_animation(animation, fig, f'reference_{operation}_animation.html')

if ANIMATE_REFERENCES:
    animate_reference('rotation')
    animate_reference('repointing')


## 7. Waypoints and executable burn accounting

Waypoints include the exact final time, even if the final interval is shorter. In the moving-reference construction, an arrival impulse and the next departure impulse share an epoch and ICRF axes. Their **vector sum** is the executable impulse. Thus $\|\Delta v_a+\Delta v_d\|$ is charged at that epoch, rather than $\|\Delta v_a\|+\|\Delta v_d\|$.

The separate arrival and departure segments make state matching easy to inspect. With zero time between them and fixed spacecraft mass, the merged impulse gives the same trajectory. All raw impulses remain available for checking. Baseline cases have departure corrections only and do not add an artificial arrival velocity correction.


In [ ]:
def waypoint_times(duration_s, spacing_s):
    if duration_s <= 0 or spacing_s <= 0:
        raise ValueError('Duration and spacing must be positive.')
    times = np.arange(0.0, duration_s, spacing_s)
    times = times[times < duration_s - 1e-7]
    return np.r_[times, duration_s]

def merge_burns(raw):
    if raw.empty:
        return pd.DataFrame(columns=['elapsed_s','dvx_mps','dvy_mps','dvz_mps','dv_mps','cumulative_mps'])
    grouped = raw.groupby('elapsed_s', sort=True)[['dvx_mps','dvy_mps','dvz_mps']].sum().reset_index()
    grouped['dv_mps'] = np.linalg.norm(grouped[['dvx_mps','dvy_mps','dvz_mps']], axis=1)
    grouped['cumulative_mps'] = grouped['dv_mps'].cumsum()
    return grouped

assert waypoint_times(10, 4).tolist() == [0,4,8,10]
trial = pd.DataFrame([[1,1,0,0], [1,-1,0,0]], columns=['elapsed_s','dvx_mps','dvy_mps','dvz_mps'])
assert merge_burns(trial)['dv_mps'].sum() == 0


## 8. Record the settings

Save a machine-readable description before propagation. Keep the embedded state epoch separate from the follower release epoch. The actual force-model component and its installed attributes are recorded after starting STK. No Δv is inferred from a reference speed or an animation.


In [ ]:
setting_names = [name for name in list(globals()) if name.isupper() and name not in
                 ['OUT', 'VERTICES', 'PAIRS', 'EDGES', 'Q0', 'SLEW_AXIS']]
settings = {'created_utc': datetime.now(timezone.utc).isoformat(), 'status': 'running'}
for name in setting_names:
    value = globals()[name]
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (str, int, float, bool, list, type(None))):
        settings[name] = value
settings['base_seed_provenance'] = 'Rounded Earth-ICRF state printed in supplied LIFE_formation_first_steps.ipynb, cell 22'
settings['base_model'] = 'Free coast from embedded state; explicit common spacecraft properties; no halo-maintenance manoeuvres'
(OUT/'settings.json').write_text(json.dumps(settings, indent=2), encoding='utf-8')
completed_cases = []


## 9. Start a fresh STK 13 Desktop scenario

Start a new Desktop application and create a new scenario. No existing STK application is attached to or closed. Set the scenario epoch to the embedded state epoch and cover the longest enabled experiment plus the release delay and a small margin.

Object Model distances and small distances are both metres, and time is seconds. This matters because Cartesian state velocity and impulsive Δv use different STK dimension combinations. `EpSec` means seconds since the **scenario epoch**, not Unix time. A new scenario therefore starts with seed EpSec = 0, even though the same physical date had a large EpSec in the old scenario.

Sources: [STK Python programming guide](https://help.agi.com/stkdevkit/Content/python/pythonProgrammingGuide.htm), [Object Model unit preferences](https://help.agi.com/stkdevkit/Content/stkUtil/introUnitPrefs.htm).


In [ ]:
if os.name != 'nt':
    raise RuntimeError('The following cells require Windows STK 13 Desktop and Astrogator.')
from agi.stk13.stkdesktop import STKDesktop
from agi.stk13.stkobjects import AgESTKObjectType, AgEVePropagatorType
from agi.stk13.stkobjects.astrogator import (
    AgEVASegmentType, AgEVAElementType, AgEVAManeuverType,
    AgEVAAttitudeControl, AgEVAControlManeuver, AgEVATargetSeqAction,
    AgEVAProfileMode,
)
stk = STKDesktop.StartApplication(visible=True, userControl=True)
root = stk.Root
root.NewScenario('LIFE_FirstSteps_' + RUN_TAG)
scenario = root.CurrentScenario
root.UnitPreferences.ResetUnits()
root.UnitPreferences.SetCurrentUnit('DateFormat', 'UTCG')
scenario.Epoch = BASE_SEED_UTCG
for dimension, value in [('Distance','m'), ('SmallDistance','m'), ('Time','sec'),
                         ('DateFormat','EpSec'), ('Mass','kg')]:
    root.UnitPreferences.SetCurrentUnit(dimension, value)
SEED_EPOCH = float(root.ConversionUtility.ConvertDate('UTCG', 'EpSec', BASE_SEED_UTCG))
T0 = SEED_EPOCH + START_OFFSET_DAYS*DAY_S
REQUIRED_DURATION_S = float((plan.duration_days.max())*DAY_S)
BASE_END_EPOCH = T0 + REQUIRED_DURATION_S + 3600.0
scenario.SetTimePeriod(str(SEED_EPOCH), str(BASE_END_EPOCH))
FRAME = 'CentralBody/Earth ICRF'
BASE_FRAME = 'CentralBody/Sun Syst_Base_L2'
print('Fresh scenario:', scenario.InstanceName)
print('Follower release UTC:', root.ConversionUtility.ConvertDate('EpSec','UTCG',str(T0)))


## 10. Create Astrogator spacecraft and coasts from explicit inputs

Insert new satellites, clear their default mission sequences, and define their initial Cartesian states. The coast uses `Earth Default High Fidelity v13`, the named propagator recorded in the earlier experiment. The notebook does not copy a coast from another spacecraft or switch silently to a two-body model. It stops if the selected component is unavailable.

Mass is held fixed during impulses. The explicit 50 kg dry mass, 50 kg fuel and 2 m² areas reproduce the follower properties printed in the earlier notebook. Base uses these same explicit values for a matched reference; its historical physical parameters were not recorded in the supplied outputs. The installed propagator attributes are saved for inspection, including numerical-integrator and force-model settings exposed by STK. Reporting cadence does not set integrator accuracy.

**Embedded Base state.** The seed in section 1 is the Earth-centred ICRF state printed at 6 October 2022, 07:10 UTC in the earlier notebook. Its printed position has limited precision (metre-to-several-metre level). Propagating this state gives an independent reference arc near the same location, but cannot reproduce the original ephemeris bit for bit. The original JWST file and full-precision Base state are absent from the supplied files. There is no reconstruction of the earlier crossing-targeted Base mission or a new halo orbit optimisation here. Base coasts without impulses; long arcs and large release delays need inspection for departure from the L2 neighbourhood. Replacing the seed with a full-precision state is sufficient to change the starting orbit without introducing a file dependency.

Sources: [Initial State](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSInitialState.html), [Propagate](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSPropagate.html), [Component Browser commands](https://help.agi.com/stk/12.7.1/Subsystems/connectCmds/Content/compBrowser.htm).


In [ ]:
def new_experiment(label, position_m, velocity_mps, start_epoch=None, frame=FRAME):
    if start_epoch is None:
        start_epoch = T0
    name = 'FS_' + label
    sat = scenario.Children.New(AgESTKObjectType.eSatellite, name)
    sat.SetPropagatorType(AgEVePropagatorType.ePropagatorAstrogator)
    driver = sat.Propagator
    driver.MainSequence.RemoveAll()
    initial = driver.MainSequence.Insert(AgEVASegmentType.eVASegmentTypeInitialState, 'Initial', '-')
    initial.CoordSystemName = frame
    initial.OrbitEpoch = str(float(start_epoch))
    initial.SetElementType(AgEVAElementType.eVAElementTypeCartesian)
    for attr, value in zip(['X','Y','Z','Vx','Vy','Vz'], np.r_[position_m,velocity_mps]):
        setattr(initial.Element, attr, float(value))
    properties = dict(Cd=CD, Cr=CR, DryMass=DRY_MASS_KG, DragArea=AREA_M2,
                      K1=1.0, K2=1.0, RadiationPressureArea=AREA_M2,
                      Ck=1.0, SolarRadiationPressureArea=AREA_M2)
    for attr, value in properties.items():
        setattr(initial.SpacecraftParameters, attr, float(value))
    for attr, value in dict(FuelMass=FUEL_MASS_KG, FuelDensity=1000.0,
                            TankPressure=5000.0, TankTemperature=293.15).items():
        setattr(initial.FuelTank, attr, value)
    return sat, driver

def add_coast(sequence, name, duration_s):
    segment = sequence.Insert(AgEVASegmentType.eVASegmentTypePropagate, name, '-')
    segment.PropagatorName = PROPAGATOR_NAME
    stops = segment.StoppingConditions
    if 'Duration' not in [stops.Item(i).Name for i in range(stops.Count)]:
        stops.Add('Duration')
    for i in range(stops.Count-1, -1, -1):
        if stops.Item(i).Name != 'Duration':
            stops.Remove(i)
    duration = stops.Item('Duration')
    duration.Active = True
    duration.Properties.Trip = float(duration_s)
    duration.Properties.Tolerance = 1e-6
    duration.Properties.Sequence = 'Stop'
    segment.MinPropagationTime = 0.0
    segment.EnableMaxPropagationTime = True
    segment.MaxPropagationTime = float(duration_s) + 60.0
    return segment

# GetValue returns the actual installed component attributes, not just its name.
component_info = root.ExecuteCommand(f'ComponentBrowser_RM */ GetValue "Propagators" "{PROPAGATOR_NAME}"')
component_text = '\n'.join(component_info.Item(i) for i in range(component_info.Count))
(OUT/'propagator_component.txt').write_text(component_text, encoding='utf-8')
print('Propagator:', PROPAGATOR_NAME)
print('Component definition saved to propagator_component.txt; inspect it alongside the STK Component Browser.')


## 11. Propagate Base and construct the rotating L2 frames

Propagate Base once from the embedded state, with no impulses during the experiment. The rotating Sun–Earth–Moon L2 system is the same built-in `SEM_L2` system used in the original setup. Assemble a second system with Base's centre as origin and those same rotating axes.

Create explicit Cartesian calculation objects in Earth ICRF and in the Base-centred rotating L2 system. Each targeter result therefore has a known origin and axes. These are created inside the fresh scenario, using the original helper's Component Browser commands directly in this cell.


In [ ]:
Base, base_driver = new_experiment('Base', BASE_SEED_R_M, BASE_SEED_V_MPS, SEED_EPOCH)
base_coast = add_coast(base_driver.MainSequence, 'Reference_coast', BASE_END_EPOCH-SEED_EPOCH)
base_driver.RunMCS()
root.ExecuteCommand('VectorTool * CentralBody/Sun Create System Syst_SEM_L2 "Assembled" "CentralBody/Sun SEM_L2" "CentralBody/Sun SEM_L2"')
root.ExecuteCommand('VectorTool * CentralBody/Sun Create System Syst_Base_L2 "Assembled" "Satellite/FS_Base Center" "CentralBody/Sun SEM_L2"')
CALC_NAMES = {}
for frame, suffix in [(FRAME, 'FS_ICRF'), (BASE_FRAME, 'FS_Base')]:
    CALC_NAMES[frame] = []
    for axis in 'XYZ':
        name = axis + '_' + suffix
        root.ExecuteCommand(f'ComponentBrowser */ Duplicate "Calculation Objects" {axis} "{name}"')
        root.ExecuteCommand(f'ComponentBrowser */ SetValue "Calculation Objects" "{name}" CoordSystem "{frame}"')
        CALC_NAMES[frame].append(name)
print('Base propagated through:', root.ConversionUtility.ConvertDate('EpSec','UTCG',str(BASE_END_EPOCH)))


## 12. Read states in explicit frames and check Base coverage

Use VGT to evaluate spacecraft centres in a named system. For inertial motion, subtract Base only after both states have been expressed in Earth-centred ICRF. Baseline positions are evaluated directly in the Base-centred rotating frame.

At impulse epochs, read MCS segment states to specify which side of the impulse is intended. Interior ephemeris samples avoid impulse epochs. Invalid states or missing time coverage stop the run.

Sources: [VGT points](https://help.agi.com/stkdevkit/Content/DocX/AgSTKVgtLib~IAgCrdnPoint.html), [state frame transformations](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAState~GetInFrameName.html).


In [ ]:
earth = root.CentralBodies.Item('Earth')
sun = root.CentralBodies.Item('Sun')
icrf_system = earth.Vgt.Systems.Item('ICRF')
l2_system = sun.Vgt.Systems.Item('Syst_Base_L2')
halo_system = sun.Vgt.Systems.Item('Syst_SEM_L2')

def xyz(vector):
    return np.array([vector.X, vector.Y, vector.Z], dtype=float)

def point_state(satellite, epoch_s, system=icrf_system):
    result = satellite.Vgt.Points.Item('Center').LocateInSystemWithRate(float(epoch_s), system)
    if not result.IsValid:
        raise RuntimeError(f'No valid state for {satellite.InstanceName} at EpSec {epoch_s}.')
    return xyz(result.Position), xyz(result.Velocity)

@lru_cache(maxsize=200000)
def base_state(epoch_s):
    return point_state(Base, float(epoch_s))

def state_vectors(state, frame=FRAME):
    converted = state.GetInFrameName(frame)
    converted.SetElementType(AgEVAElementType.eVAElementTypeCartesian)
    e = converted.Element
    position = np.array([e.X,e.Y,e.Z], dtype=float)
    velocity = np.array([e.Vx,e.Vy,e.Vz], dtype=float)
    if not np.isfinite(np.r_[position,velocity]).all():
        raise RuntimeError('STK returned a non-finite state in ' + frame)
    return position, velocity

def segment_state(segment, frame=FRAME):
    return state_vectors(segment.FinalState, frame)

for epoch in [SEED_EPOCH, T0, T0+REQUIRED_DURATION_S, BASE_END_EPOCH]:
    base_state(float(epoch))
np.testing.assert_allclose(base_state(SEED_EPOCH)[0], BASE_SEED_R_M, rtol=0, atol=1e-3)
np.testing.assert_allclose(base_state(SEED_EPOCH)[1], BASE_SEED_V_MPS, rtol=0, atol=VELOCITY_TOL_MPS)
print('Base start [m], [m/s]:', base_state(T0))


## 13. Inspect the independent Base reference arc

Plot Base relative to the L2 origin, in rotating L2 axes, over the complete generated arc. The release point is marked separately if a delay was chosen. A short arc need not show a closed halo shape. This plot helps detect implausible long-arc drift before interpreting formation results; it is not a proof of a periodic halo orbit.


In [ ]:
base_times = np.linspace(SEED_EPOCH, BASE_END_EPOCH, 401)
base_path = np.array([point_state(Base, t, halo_system)[0] for t in base_times])
base_export = pd.DataFrame(base_path, columns=['x_m','y_m','z_m'])
base_export.insert(0, 'epoch_EpSec', base_times)
base_export.to_csv(OUT/'base_reference.csv', index=False)
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(projection='3d')
ax.plot(*(base_path/1000).T, label='Base coast')
ax.scatter(*point_state(Base,T0,halo_system)[0]/1000, color='darkred', label='Follower release')
ax.scatter(0,0,0,marker='+',color='black',label='L2 origin')
ax.set(xlabel='L2 X [km]',ylabel='L2 Y [km]',zlabel='L2 Z [km]',title='Independent Base reference arc')
ax.legend(); fig.tight_layout()
fig.savefig(OUT/'base_reference.png',dpi=180,bbox_inches='tight'); plt.show()


## 14. Configure impulses and the differential corrector

Each target sequence contains a departure impulse and a coast. The three ICRF Cartesian impulse components are controls. Coast-end position coordinates are the three constraints, expressed in the requested target frame. Baseline targets use rotating Base-centred coordinates; moving-reference targets use Earth-centred ICRF.

An arrival impulse, when requested, is computed separately after the departure correction converges. A nominal frame-and-unit check compares the calculation-object values with the coast's transformed final state before targeting.

Sources: [Target Sequence](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSTargetSequence.html), [impulsive thrust vector](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAAttitudeControlImpulsiveThrustVector.html), [Differential Corrector](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAProfileDifferentialCorrector.html).


In [ ]:
def add_impulse(sequence, name, dv_mps=(0,0,0)):
    segment = sequence.Insert(AgEVASegmentType.eVASegmentTypeManeuver, name, '-')
    segment.SetManeuverType(AgEVAManeuverType.eVAManeuverTypeImpulsive)
    segment.Maneuver.SetAttitudeControlType(AgEVAAttitudeControl.eVAAttitudeControlThrustVector)
    segment.Maneuver.UpdateMass = False
    segment.Maneuver.AttitudeControl.ThrustAxesName = FRAME
    segment.Maneuver.AttitudeControl.AssignCartesian(*map(float, dv_mps))
    return segment

def make_target_leg(driver, leg_number, duration_s, desired_position_m, frame, tolerance_m):
    ts = driver.MainSequence.Insert(AgEVASegmentType.eVASegmentTypeTargetSequence, f'Target_{leg_number:03d}', '-')
    ts.ContinueOnFailure = False
    departure = add_impulse(ts.Segments, 'Departure')
    coast = add_coast(ts.Segments, 'Coast', duration_s)
    dc = ts.Profiles.Item('Differential Corrector')
    dc.Mode = AgEVAProfileMode.eVAProfileModeIterate
    dc.MaxIterations = DC_MAX_ITERATIONS
    dc.EnableDisplayStatus = False
    controls = [AgEVAControlManeuver.eVAControlManeuverImpulsiveCartesianX,
                AgEVAControlManeuver.eVAControlManeuverImpulsiveCartesianY,
                AgEVAControlManeuver.eVAControlManeuverImpulsiveCartesianZ]
    for axis, control_enum in zip('XYZ', controls):
        departure.EnableControlParameter(control_enum)
        control = dc.ControlParameters.GetControlByPaths('Departure', f'ImpulsiveMnvr.Cartesian.{axis}')
        control.Enable = True
        control.Perturbation = DC_PERTURBATION_MPS
        control.MaxStep = DC_MAX_STEP_MPS
    for name, desired in zip(CALC_NAMES[frame], desired_position_m):
        coast.Results.Add('Cartesian Elems/' + name)
        result = dc.Results.GetResultByPaths('Coast', name)
        result.Enable = True
        result.DesiredValue = float(desired)
        result.Tolerance = tolerance_m
    ts.Action = AgEVATargetSeqAction.eVATargetSeqActionRunNominalSeq
    return ts, departure, coast, dc


## 15. Solve and verify one leg

Run the differential corrector, check its component residuals, apply the converged controls, and rerun nominally. Verify that the actual departure velocity jump equals the impulse in m/s and that the coast ends at the intended epoch.

For a moving-reference transfer, add $\Delta v_a=v_{\rm desired}-v_{\rm coast,end}$ in ICRF, then verify the complete final state. For the position-only baseline, `desired_v=None` adds no arrival burn and leaves arrival velocity unconstrained. The next leg inherits that actual velocity. No leg resets the spacecraft to an ideal waypoint.

The position tolerance applies to each coordinate; the corresponding norm bound is √3 times that value. A failure stops the experiment instead of adding an unconverged case to the comparison.


In [ ]:
def solve_leg(driver, number, dt, desired_r, desired_v=None, frame=FRAME, tolerance_m=POSITION_TOL_M):
    ts, departure, coast, dc = make_target_leg(driver, number, dt, desired_r, frame, tolerance_m)
    driver.RunMCS()
    nominal_r, _ = segment_state(coast, frame)
    calc_r = np.array([float(coast.GetResultValue(name)) for name in CALC_NAMES[frame]])
    np.testing.assert_allclose(calc_r, nominal_r, rtol=0, atol=0.005,
                               err_msg='Target calculation frame or units do not match the requested state frame.')
    ts.Action = AgEVATargetSeqAction.eVATargetSeqActionRunActiveProfiles
    driver.RunMCS()
    diagnostic = {'leg':number, 'profile_status':str(dc.Status), 'iterations':int(dc.NumIterations)}
    for axis, name in zip('XYZ', CALC_NAMES[frame]):
        result = dc.Results.GetResultByPaths('Coast', name)
        residual = float(result.CurrentValue-result.DesiredValue)
        diagnostic[f'target_residual_{axis}_m'] = residual
        if not np.isfinite(residual) or abs(residual) > tolerance_m*1.01:
            raise RuntimeError(f'Position target failed: {diagnostic}')
    ts.ApplyProfiles()
    ts.Action = AgEVATargetSeqAction.eVATargetSeqActionRunNominalSeq
    driver.RunMCS()
    r_end, _ = segment_state(coast, frame)
    if np.max(np.abs(r_end-desired_r)) > tolerance_m*1.05:
        raise RuntimeError('Applied nominal solution misses the position target.')
    dv_departure = np.array(departure.Maneuver.AttitudeControl.QueryCartesian(), dtype=float).ravel()
    velocity_before = state_vectors(departure.InitialState)[1]
    np.testing.assert_allclose(segment_state(departure)[1]-velocity_before, dv_departure,
                               rtol=0, atol=VELOCITY_TOL_MPS)
    expected_epoch = float(coast.InitialState.Epoch)+dt
    if abs(float(coast.FinalState.Epoch)-expected_epoch) > 1e-3:
        raise RuntimeError('Coast stopped at an unexpected epoch.')
    diagnostic['endpoint_position_error_m'] = float(np.linalg.norm(r_end-desired_r))
    arrival, dv_arrival = None, np.zeros(3)
    diagnostic['pre_arrival_velocity_error_mps'] = np.nan
    diagnostic['endpoint_velocity_error_mps'] = np.nan
    if desired_v is not None:
        # Velocity matching is always performed in Earth ICRF, regardless of position frame.
        v_end = segment_state(coast)[1]
        dv_arrival = desired_v-v_end
        arrival = add_impulse(driver.MainSequence, f'Arrival_{number:03d}', dv_arrival)
        driver.RunMCS()
        achieved_r = segment_state(arrival, frame)[0]
        achieved_v = segment_state(arrival)[1]
        if (np.max(np.abs(achieved_r-desired_r)) > tolerance_m*1.05
                or np.linalg.norm(achieved_v-desired_v) > VELOCITY_TOL_MPS):
            raise RuntimeError('Post-arrival state verification failed.')
        diagnostic['pre_arrival_velocity_error_mps'] = float(np.linalg.norm(v_end-desired_v))
        diagnostic['endpoint_velocity_error_mps'] = float(np.linalg.norm(achieved_v-desired_v))
    return departure, coast, arrival, dv_departure, dv_arrival, diagnostic


## 16. Common sampling, exports and progress checkpoints

Take the exact post-departure and pre-arrival coast states from their segments and interior samples from VGT. The interior grid is anchored to the start of the experiment and includes at least a midpoint in every coast. This gives consistent sampling across spacecraft and still shows short intervals if the reporting cadence is coarse.

Store raw impulses, merged executable impulses, trajectories, diagnostics and a summary for each case. RMS error is integrated over elapsed time, so duplicated coast-boundary rows do not overweight waypoints. Maximum error remains a sampled estimate. The summary distinguishes baseline correction interval from moving-reference burn spacing.


In [ ]:
def coast_samples(sat, departure, coast, ta, tb, frame=FRAME):
    first_index = max(0, int(np.floor(ta/SAMPLE_S - 0.5)) + 1)
    interior = (np.arange(first_index, int(np.ceil(tb/SAMPLE_S))) + 0.5)*SAMPLE_S
    interior = interior[(interior > ta+1e-5) & (interior < tb-1e-5)]
    interior = np.unique(np.r_[interior, (ta+tb)/2])
    samples = [(float(ta), *segment_state(departure, frame), 'post_departure')]
    system = icrf_system if frame == FRAME else l2_system
    for t in interior:
        samples.append((float(t), *point_state(sat,T0+float(t),system), 'coast'))
    samples.append((float(tb), *segment_state(coast, frame), 'pre_arrival'))
    return samples

def save_case(label, operation, vertex, duration_s, spacing_s, sat, driver, samples, burns, diagnostics):
    raw = pd.DataFrame(burns, columns=['elapsed_s','kind','dvx_mps','dvy_mps','dvz_mps'])
    raw['dv_mps'] = np.linalg.norm(raw[['dvx_mps','dvy_mps','dvz_mps']], axis=1)
    merged = merge_burns(raw)
    trajectory = pd.DataFrame(samples, columns=['epoch_EpSec','elapsed_s','side','x_m','y_m','z_m',
        'ref_x_m','ref_y_m','ref_z_m','vx_mps','vy_mps','vz_mps','ex_m','ey_m','ez_m','error_m'])
    diagnostic_table = pd.DataFrame(diagnostics)
    for suffix, table in [('trajectory',trajectory),('raw_burns',raw),('burns',merged),('targeting',diagnostic_table)]:
        table.to_csv(OUT/f'{label}_{suffix}.csv', index=False)
    unique = trajectory.sort_values('elapsed_s').drop_duplicates('elapsed_s')
    rms = np.sqrt(trapezoid(unique.error_m.to_numpy()**2, unique.elapsed_s.to_numpy())/duration_s)
    summary = dict(case=label, operation=operation, vertex=LABELS[vertex], duration_s=float(duration_s),
                   spacing_s=float(spacing_s),
                   sk_interval_days=spacing_s/DAY_S if operation == 'baseline_L2' else None,
                   burn_spacing_hours=spacing_s/3600 if operation != 'baseline_L2' else None,
                   sampled_max_error_m=float(trajectory.error_m.max()), rms_error_m=float(rms),
                   dv_mps=float(merged.dv_mps.sum()), unmerged_dv_mps=float(raw.dv_mps.sum()),
                   first_burn_mps=float(merged.dv_mps.iloc[0]), last_burn_mps=float(merged.dv_mps.iloc[-1]),
                   executable_burn_epochs=int(len(merged)),
                   position_frame='Base rotating L2' if operation == 'baseline_L2' else 'Base-centred ICRF',
                   stop_at_end=bool(STOP_AT_END) if operation not in ['baseline_L2','single_transfer'] else False)
    (OUT/f'{label}_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    case = dict(satellite=sat, driver=driver, trajectory=trajectory, raw_burns=raw,
                burns=merged, targeting=diagnostic_table, summary=summary)
    completed_cases.append(case)
    pd.DataFrame([c['summary'] for c in completed_cases]).to_csv(OUT/'run_summary.csv',index=False)
    return case

def number_tag(value):
    return format(float(value), '.12g').replace('.', 'p')


## 17. Fixed-offset station keeping for every selected interval

For each selected follower and each `SK_INTERVALS_DAYS` value, start at its oriented octahedron offset, fixed in Base-centred rotating L2 coordinates. Initial velocity is zero **in those rotating coordinates**, which includes the appropriate frame rotation when transformed to ICRF. The baseline uses the same initial roll construction, interpreted in L2 axes, rather than the inertial orientation of the later rotation.

Target the same relative position at the end of each interval with a departure correction only, as in the original helper. Use separate, applied target sequences so that every interval has its own saved correction and convergence evidence. If the common duration ends inside an interval, target that shortened last leg; this keeps comparisons at the same final epoch. There is no terminal stop or arrival velocity constraint.

This is the original position-only baseline method reconstructed around the independent Base arc. It is not an exact replay of the old repeated sequence, its full 180-day horizon, or the inherited physical state. The old Follow offsets were in the leader body frame; here their intended L2 interpretation is made explicit at release. Every burn included in the total has an actual elapsed timestamp.


In [ ]:
def run_baseline(label, vertex, spacing_s):
    offset = (VERTICES @ Q0.T)[vertex]
    sat, driver = new_experiment(label, offset, np.zeros(3), frame=BASE_FRAME)
    times = waypoint_times(BASELINE_DURATION_S, spacing_s)
    samples, burns, diagnostics = [], [], []
    for leg, (ta,tb) in enumerate(zip(times[:-1],times[1:]), 1):
        departure, coast, _, dv0, _, diag = solve_leg(
            driver, leg, tb-ta, offset, desired_v=None, frame=BASE_FRAME,
            tolerance_m=BASELINE_POSITION_TOL_M)
        diagnostics.append(dict(diag, elapsed_s=float(tb)))
        burns.append([float(ta),'departure',*dv0])
        for t, rel_r, rel_v, side in coast_samples(sat,departure,coast,ta,tb,BASE_FRAME):
            error = rel_r-offset
            samples.append([T0+t,t,side,*rel_r,*offset,*rel_v,*error,np.linalg.norm(error)])
        print(f'{label}: leg {leg}/{len(times)-1}; endpoint error {diag["endpoint_position_error_m"]:.3g} m')
    return save_case(label,'baseline_L2',vertex,BASELINE_DURATION_S,spacing_s,
                     sat,driver,samples,burns,diagnostics)

baseline_cases = []
if RUN_BASELINE:
    for interval in SK_INTERVALS_DAYS:
        for vertex in SELECTED_VERTICES:
            label = f'baseline_{number_tag(interval)}d_{LABELS[vertex]}'
            baseline_cases.append(run_baseline(label, vertex, interval*DAY_S))


## 18. Plot each case and compare baseline corrections

Plot desired and propagated paths, tracking error, executable impulse magnitudes, and cumulative Δv. Baseline axes are rotating L2 axes; later manoeuvre plots use ICRF directions. The stem plot shows actual burn epochs, including any initial correction. The full-window total is the sum of the displayed executable impulses.

Several baseline intervals use the same observation window. A 14-day baseline over 21 days has a shortened final seven-day leg; the boundary condition therefore matters when interpreting its cost.


In [ ]:
def plot_case(case):
    tr, b = case['trajectory'], case['burns']
    summary = case['summary']
    operation, duration = summary['operation'], summary['duration_s']
    vertex = LABELS.index(summary['vertex'])
    if operation == 'baseline_L2':
        desired = tr[['ref_x_m','ref_y_m','ref_z_m']].to_numpy()
        axis_name = 'L2'
    else:
        reference_operation = 'rotation' if operation == 'single_transfer' else operation
        desired = np.array([reference(t,reference_operation)[0][vertex] for t in np.linspace(0,duration,401)])
        axis_name = 'ICRF'
    fig = plt.figure(figsize=(12,9))
    ax = fig.add_subplot(221,projection='3d')
    ax.plot(*desired.T,'--',label='desired'); ax.plot(tr.x_m,tr.y_m,tr.z_m,label='actual')
    ax.set(xlabel=f'{axis_name} X [m]',ylabel=f'{axis_name} Y [m]',zlabel=f'{axis_name} Z [m]'); ax.legend()
    ax = fig.add_subplot(222)
    ax.plot(tr.elapsed_s/DAY_S,tr.error_m)
    ax.set(xlabel='Elapsed time [days]',ylabel='Position tracking error [m]'); ax.grid(alpha=.3)
    ax = fig.add_subplot(223)
    ax.stem(b.elapsed_s/DAY_S,b.dv_mps)
    ax.set(xlabel='Elapsed time [days]',ylabel='Executable impulse [m/s]'); ax.grid(alpha=.3)
    ax = fig.add_subplot(224)
    ax.step(b.elapsed_s/DAY_S,b.cumulative_mps,where='post')
    ax.set(xlabel='Elapsed time [days]',ylabel='Cumulative Δv [m/s]'); ax.grid(alpha=.3)
    fig.suptitle(summary['case']); fig.tight_layout()
    fig.savefig(OUT/f'{summary["case"]}_plots.png',dpi=180,bbox_inches='tight'); plt.show()

for case in baseline_cases:
    plot_case(case)
if baseline_cases:
    display(pd.DataFrame([c['summary'] for c in baseline_cases])[
        ['vertex','sk_interval_days','dv_mps','sampled_max_error_m','rms_error_m']])


## 19. Track a moving reference with the same two-impulse building block

Start at the desired position with zero relative ICRF velocity, so the first departure includes starting the rotation. At each next waypoint, target Base position plus the desired relative position. Then add an arrival impulse to match Base velocity plus the desired relative velocity. The next leg inherits the actual achieved state.

If `STOP_AT_END=True`, the final arrival instead matches Base velocity: stopping is included in the final impulse. `False` finishes with the desired moving-reference velocity. The single-transfer demonstration always matches the moving velocity. The matched inertial hold uses exactly the same duration, waypoint spacing, geometry, epoch, physical properties and targeting construction as the rotation.


In [ ]:
def run_reference(label, operation, vertex, duration_s, spacing_s, stop_at_end=STOP_AT_END,
                  summary_operation=None):
    times = waypoint_times(duration_s, spacing_s)
    base_r0, base_v0 = base_state(float(T0))
    p0, _ = reference(0, operation)
    sat, driver = new_experiment(label, base_r0+p0[vertex], base_v0)
    samples, burns, diagnostics = [], [], []
    for leg, (ta,tb) in enumerate(zip(times[:-1],times[1:]), 1):
        br, bv = base_state(float(T0+tb))
        desired_p, desired_v = reference(tb, operation)
        target_v = bv+desired_v[vertex]
        if stop_at_end and leg == len(times)-1:
            target_v = bv.copy()
        departure, coast, arrival, dv0, dv1, diag = solve_leg(
            driver,leg,tb-ta,br+desired_p[vertex],target_v)
        diagnostics.append(dict(diag,elapsed_s=float(tb)))
        burns.extend([[float(ta),'departure',*dv0], [float(tb),'arrival',*dv1]])
        for t, actual_r, actual_v, side in coast_samples(sat,departure,coast,ta,tb):
            base_r, base_v = base_state(float(T0+t))
            ref_p, _ = reference(t,operation)
            rel_r, rel_v = actual_r-base_r, actual_v-base_v
            error = rel_r-ref_p[vertex]
            samples.append([T0+t,t,side,*rel_r,*ref_p[vertex],*rel_v,*error,np.linalg.norm(error)])
        print(f'{label}: leg {leg}/{len(times)-1}; endpoint error {diag["endpoint_position_error_m"]:.3g} m; DC {diag["profile_status"]}')
    case = save_case(label,summary_operation or operation,vertex,duration_s,spacing_s,
                     sat,driver,samples,burns,diagnostics)
    case['summary']['stop_at_end'] = bool(stop_at_end)
    return case


## 20. Demonstrate one nearby-state transfer

Run one leg for the first selected spacecraft, using the largest selected burn spacing. It starts stationary relative to Base in inertial axes and ends with the correct rotation velocity. Inspect the velocity error before the arrival burn and its residual afterward: position targeting alone cannot impose a desired moving-reference velocity.

This is an additional diagnostic spacecraft; it is not counted again in the later formation totals.


In [ ]:
single = None
if RUN_SINGLE_TRANSFER:
    single = run_reference('single_transfer','rotation',SELECTED_VERTICES[0],
                           DEMO_SPACING_S,DEMO_SPACING_S,stop_at_end=False,
                           summary_operation='single_transfer')
    display(single['targeting'])
    plot_case(single)


## 21. Run every selected rotation spacing and matched hold

Run each selected follower independently for every chosen burn spacing. An internal arrival and departure will later be charged as their vector sum. The resulting formation contains exactly the selected labels for each spacing, rather than repeatedly simulating only +X and multiplying its cost by six.

With `[3, 6]` hours, the 3-hour case is the first waypoint-refinement comparison of the 6-hour case. Add another spacing to check whether the trend stabilises. The matched hold is separate from the rotating-L2 position-only baseline; subtract costs only after matching duration, spacing and start/stop conventions.


In [ ]:
rotation_cases, hold_cases = [], []
for spacing_hr in BURN_SPACINGS_HOURS:
    for vertex in SELECTED_VERTICES:
        suffix = f'{number_tag(spacing_hr)}h_{LABELS[vertex]}'
        if RUN_ROTATION:
            rotation_cases.append(run_reference('rotation_'+suffix,'rotation',vertex,
                                                 TRACK_DURATION_S,spacing_hr*3600))
        if RUN_MATCHED_HOLD:
            hold_cases.append(run_reference('hold_'+suffix,'inertial_hold',vertex,
                                             TRACK_DURATION_S,spacing_hr*3600))


## 22. Compare Δv, tracking errors and per-spacecraft expenditure

Save each case's four-panel plot and compare the spacing dependence. Formation totals sum the actual selected followers; they do not extrapolate a single follower. The worst sampled tracking error is the largest among those followers.

The rotation-minus-hold quantity is a difference between two matched experiments. It is not a unique decomposition into independent rotation and maintenance fuel, because the corrections combine as vectors and the trajectories differ. A smaller waypoint error does not establish a smaller inter-waypoint error; inspect both the endpoint diagnostics and full error curves.


In [ ]:
for case in rotation_cases+hold_cases:
    plot_case(case)
moving_summary = pd.DataFrame([c['summary'] for c in rotation_cases+hold_cases])
if not moving_summary.empty:
    display(moving_summary[['operation','vertex','burn_spacing_hours','dv_mps','sampled_max_error_m','rms_error_m']])
    formation_totals = moving_summary.groupby(['operation','burn_spacing_hours'],as_index=False).agg(
        spacecraft=('vertex','count'), total_dv_mps=('dv_mps','sum'),
        worst_sampled_error_m=('sampled_max_error_m','max'))
    formation_totals.to_csv(OUT/'moving_formation_totals.csv',index=False)
    display(formation_totals)
    fig,axes = plt.subplots(1,2,figsize=(12,4))
    for (operation,vertex), group in moving_summary.groupby(['operation','vertex']):
        group = group.sort_values('burn_spacing_hours')
        label = f'{operation} {vertex}'
        axes[0].plot(group.burn_spacing_hours,group.dv_mps,'o-',label=label)
        axes[1].plot(group.burn_spacing_hours,group.sampled_max_error_m,'o-',label=label)
    axes[0].set(xlabel='Burn spacing [h]',ylabel='Executable Δv [m/s]')
    axes[1].set(xlabel='Burn spacing [h]',ylabel='Sampled maximum error [m]')
    for ax in axes: ax.grid(alpha=.3)
    axes[0].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT/'spacing_comparison.png',dpi=180,bbox_inches='tight'); plt.show()
    if rotation_cases and hold_cases:
        matched = moving_summary.pivot(index=['vertex','burn_spacing_hours'],columns='operation',values='dv_mps')
        matched['rotation_minus_hold_mps'] = matched['rotation']-matched['inertial_hold']
        matched.to_csv(OUT/'matched_cost_comparison.csv')
        display(matched)


## 23. Animate the propagated spacecraft and tracking errors

Animate one common-spacing set of actual propagated followers beside the complete desired octahedron. Each selected follower has a trail; a line joins its actual position to its desired vertex. The right panel shows its error history with a moving time cursor. Hollow unselected vertices remain reference-only and are never presented as propagated spacecraft.

Playback positions are linearly interpolated between saved position samples for smooth display. Error statistics come from the original samples, not these playback frames. The animation does not magnify the physical displacement; the error plot makes small differences visible. Reduce `SAMPLE_S` if coast shapes or error peaks are not resolved. One animation is produced for each selected rotation spacing, enabling a direct visual refinement comparison.


In [ ]:
def animation_positions(cases, times):
    positions = []
    for case in cases:
        tr = case['trajectory'].sort_values('elapsed_s').drop_duplicates('elapsed_s')
        positions.append(np.column_stack([np.interp(times,tr.elapsed_s,tr[axis+'_m']) for axis in 'xyz']))
    return np.stack(positions,axis=1)

def animate_actual(cases, operation, filename):
    if not cases:
        return
    duration = cases[0]['summary']['duration_s']
    times = np.linspace(0,duration,ANIMATION_FRAMES)
    actual = animation_positions(cases,times)
    vertices = [LABELS.index(c['summary']['vertex']) for c in cases]
    desired = np.array([reference(t,operation)[0] for t in times])
    limit = 1.15*max(L_M,float(np.max(np.abs(actual))))
    fig = plt.figure(figsize=(12,6))
    ax = fig.add_subplot(121,projection='3d'); error_ax = fig.add_subplot(122)
    fig.subplots_adjust(left=.03,right=.98,bottom=.12,top=.90,wspace=.48)
    ax.set(xlim=(-limit,limit),ylim=(-limit,limit),zlim=(-limit,limit),
           xlabel='ICRF X [m]',ylabel='ICRF Y [m]',zlabel='ICRF Z [m]')
    ax.set_box_aspect((1,1,1))
    edges = [ax.plot([],[],[],color='steelblue',alpha=.35)[0] for _ in EDGES]
    reference_points = ax.scatter([],[],[],facecolors='none',edgecolors='grey',label='Desired six vertices')
    actual_points = ax.scatter([],[],[],color='darkred',label='Actual selected followers')
    trails = [ax.plot([],[],[],alpha=.7,label=c['summary']['vertex'])[0] for c in cases]
    links = [ax.plot([],[],[],color='darkred',linestyle=':')[0] for _ in cases]
    ax.scatter(0,0,0,color='black',marker='+',label='Base'); ax.legend(fontsize=7)
    for c in cases:
        tr = c['trajectory']
        error_ax.plot(tr.elapsed_s/DAY_S,tr.error_m,label=c['summary']['vertex'])
    cursor = error_ax.axvline(0,color='black',linestyle=':')
    error_ax.set(xlabel='Elapsed time [days]',ylabel='Tracking error [m]')
    error_ax.grid(alpha=.3); error_ax.legend(fontsize=8)

    def draw_frame(k):
        for edge,(i,j) in zip(edges,EDGES): edge.set_data_3d(*desired[k,[i,j]].T)
        reference_points._offsets3d = tuple(desired[k].T)
        actual_points._offsets3d = tuple(actual[k].T)
        for j,(trail,link,vertex) in enumerate(zip(trails,links,vertices)):
            trail.set_data_3d(*actual[:k+1,j].T)
            link.set_data_3d(*np.array([actual[k,j],desired[k,vertex]]).T)
        cursor.set_xdata([times[k]/DAY_S,times[k]/DAY_S])
        ax.set_title(f'{operation}, {cases[0]["summary"]["burn_spacing_hours"]:g} h spacing: {times[k]/DAY_S:.2f} d')
        return edges+trails+links+[cursor]

    animation = FuncAnimation(fig,draw_frame,frames=len(times),interval=ANIMATION_INTERVAL_MS)
    show_animation(animation,fig,filename)

if ANIMATE_ACTUAL:
    for spacing_hr in BURN_SPACINGS_HOURS:
        group = [c for c in rotation_cases if c['summary']['spacing_s'] == spacing_hr*3600]
        animate_actual(group,'rotation',f'actual_rotation_{number_tag(spacing_hr)}h_animation.html')


## 24. Check the sampled formation geometry

For two or more propagated followers, compare their actual pair separations with their desired pair separations at common saved epochs. This detects distortion between targeted waypoints. Only selected followers are included; a subset is not labelled a complete octahedron.

The minimum sampled separation is a diagnostic, not a collision-clearance certificate. No continuous-time bound, spacecraft size, navigation uncertainty or collision-avoidance constraint is imposed. With one follower, pairwise geometry is undefined and this check is skipped.


In [ ]:
def check_formation(cases, operation, label):
    if len(cases) < 2:
        print(label + ': select at least two followers for pair-separation diagnostics.')
        return
    vertices = [LABELS.index(c['summary']['vertex']) for c in cases]
    tables = [c['trajectory'].sort_values('elapsed_s').drop_duplicates('elapsed_s').set_index('elapsed_s') for c in cases]
    # These cases share a waypoint and reporting grid; use actual common samples, without interpolation.
    times = tables[0].index.to_numpy()
    for table in tables[1:]: times = np.intersect1d(times,table.index.to_numpy())
    actual = np.stack([table.loc[times,['x_m','y_m','z_m']].to_numpy() for table in tables],axis=1)
    desired = np.array([reference(t,operation)[0][vertices] for t in times])
    rows = []
    for i,j in combinations(range(len(cases)),2):
        distance = np.linalg.norm(actual[:,i]-actual[:,j],axis=1)
        expected = np.linalg.norm(desired[:,i]-desired[:,j],axis=1)
        for t,d,e in zip(times,distance,expected):
            rows.append([t,LABELS[vertices[i]]+'-'+LABELS[vertices[j]],d,e,d-e])
    table = pd.DataFrame(rows,columns=['elapsed_s','pair','distance_m','reference_distance_m','distance_error_m'])
    table.to_csv(OUT/f'{label}_pair_separations.csv',index=False)
    stats = table.groupby('pair').agg(min_sampled_distance_m=('distance_m','min'),
                                     max_sampled_abs_error_m=('distance_error_m',lambda x: np.max(np.abs(x))))
    stats.to_csv(OUT/f'{label}_pair_summary.csv'); display(stats)
    fig,ax = plt.subplots()
    for pair,group in table.groupby('pair'):
        ax.plot(group.elapsed_s/DAY_S,group.distance_error_m,label=pair)
    ax.set(xlabel='Elapsed time [days]',ylabel='Pair separation error [m]',title=label)
    ax.grid(alpha=.3); ax.legend(fontsize=7,ncol=3)
    fig.tight_layout(); fig.savefig(OUT/f'{label}_pair_errors.png',dpi=180,bbox_inches='tight'); plt.show()

for spacing_hr in BURN_SPACINGS_HOURS:
    group = [c for c in rotation_cases if c['summary']['spacing_s'] == spacing_hr*3600]
    check_formation(group,'rotation','rotation_'+number_tag(spacing_hr)+'h')


## 25. Optional dynamical re-pointing experiment

If enabled in section 1, follow the quintic re-pointing reference for every selected follower and burn spacing. The initial and final reference angular velocities are zero. The same targeter, exports, cost accounting, animations and pairwise geometry checks apply.

The reference animation in section 6 is available even when this dynamical stage is disabled. Re-pointing duration and angle are separate from rotation period and tracked rotation duration.


In [ ]:
repointing_cases = []
if RUN_REPOINTING:
    for spacing_hr in BURN_SPACINGS_HOURS:
        group = []
        for vertex in SELECTED_VERTICES:
            label = f'repointing_{number_tag(spacing_hr)}h_{LABELS[vertex]}'
            case = run_reference(label,'repointing',vertex,SLEW_DURATION_S,spacing_hr*3600)
            group.append(case); repointing_cases.append(case); plot_case(case)
        check_formation(group,'repointing','repointing_'+number_tag(spacing_hr)+'h')
        if ANIMATE_ACTUAL:
            animate_actual(group,'repointing',f'actual_repointing_{number_tag(spacing_hr)}h_animation.html')
else:
    print('Dynamical re-pointing disabled in section 1; the desired motion is still defined and animated.')


## 26. Final summaries and scenario checkpoint

Write summaries for every completed case and totals grouped by operation, duration and spacing. The diagnostic single-transfer case is excluded from formation totals. Save a new scenario in this run's results directory, leaving the Desktop open for inspection. The checkpoint and exports are outputs, never input dependencies of the notebook.

Use `*_burns.csv` for executable Δv and `*_raw_burns.csv` for the two-impulse construction. `*_targeting.csv` contains endpoint checks; `*_trajectory.csv` contains sampled inter-waypoint errors. `run_plan.csv` and `settings.json` identify exactly what was simulated. Saved HTML animations can be opened independently of STK.


In [ ]:
summary = pd.DataFrame([case['summary'] for case in completed_cases])
summary.to_csv(OUT/'run_summary.csv',index=False)
formation_summary = summary[summary.operation != 'single_transfer'].groupby(
    ['operation','duration_s','spacing_s'],as_index=False).agg(
        spacecraft=('vertex','count'),total_dv_mps=('dv_mps','sum'),
        worst_sampled_error_m=('sampled_max_error_m','max'))
formation_summary.to_csv(OUT/'formation_totals.csv',index=False)
settings['release_UTCG'] = root.ConversionUtility.ConvertDate('EpSec','UTCG',str(T0))
settings['completed_cases'] = len(completed_cases)
settings['status'] = 'complete'
(OUT/'settings.json').write_text(json.dumps(settings,indent=2),encoding='utf-8')
display(summary[['case','dv_mps','sampled_max_error_m','rms_error_m']])
display(formation_summary)
if SAVE_SCENARIO:
    scenario_path = (OUT/'STK_scenario'/'LIFE_FirstSteps.sc').resolve()
    scenario_path.parent.mkdir(exist_ok=True)
    root.SaveScenarioAs(str(scenario_path))
    print('Saved scenario:',scenario_path)
root.Rewind()
root.CurrentTime = T0
print('Completed outputs:',OUT)


## Reading the experiment and deciding the next step

- **Where the experiment starts:** Base begins from the embedded 6 October 2022 state. Followers start at that epoch plus `START_OFFSET_DAYS`, already placed at their reference offsets. Initial deployment or transfer from Earth/JWST is outside the Δv budget.
- **What the time settings do:** baseline duration is the comparison window; station-keeping interval is its position-targeting horizon; rotation period specifies a complete reference turn; tracked duration specifies how much of it is simulated; burn spacing specifies the moving-reference waypoint grid. None is inherited from a previous notebook.
- **What can be concluded:** compare sampled path error and merged Δv versus spacing and spacecraft label. The two on-axis reference vertices do not rotate, but still need maintenance. Actual formation geometry need not remain rigid between burns.
- **What cannot yet be concluded:** a low numerical endpoint residual does not prove acceptable continuous tracking, collision clearance, optimal Δv, robustness to navigation errors, or a physically justified LIFE observation schedule.

First inspect the fresh Base arc, force-model component and targeter diagnostics. Reduce the reporting cadence to check error peaks; separately tighten the numerical integrator in a copied propagator component and repeat a small case before trusting differences comparable with numerical accuracy. Compare at least two, preferably three, burn spacings. Then run all six followers over one complete rotation and, separately, one slew. Study formation size, line-of-sight direction and release delay one at a time. Long delays require checking that the freely propagated Base still represents a useful L2 reference; a sustained halo-phase study would need a suitable maintained or periodic reference trajectory.

## Official Ansys STK documentation

The notebook retains `agi.stk13` and the original Object Model method names. PySTK's `ansys.stk.core` package uses a different naming convention and is not mixed into this code.

- [Python programming guide](https://help.agi.com/stkdevkit/Content/python/pythonProgrammingGuide.htm): Desktop application and Object Model access.
- [Object Model unit preferences](https://help.agi.com/stkdevkit/Content/stkUtil/introUnitPrefs.htm): Distance, SmallDistance, Time, Mass and DateFormat.
- [Initial State](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSInitialState.html): Cartesian state, epoch, spacecraft parameters and tank values.
- [Propagate](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSPropagate.html) and [stopping conditions](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAStoppingConditionCollection.html): selected propagator and Duration stop.
- [Target Sequence](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSTargetSequence.html): controls, constraints and ApplyProfiles.
- [Differential Corrector](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAProfileDifferentialCorrector.html) and [DC results](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVADCResult.html): iteration limits, current values and tolerances.
- [Impulsive thrust vector](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAAttitudeControlImpulsiveThrustVector.html): Cartesian Δv, SmallDistance/Time units, and thrust axes.
- [State frame transformation](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAState~GetInFrameName.html) and [VGT points](https://help.agi.com/stkdevkit/Content/DocX/AgSTKVgtLib~IAgCrdnPoint.html): explicit origin, axes and position/velocity evaluation.
- [Component Browser Connect](https://help.agi.com/stk/12.7.1/Subsystems/connectCmds/Content/compBrowser.htm): duplicate calculation objects, set their coordinate systems, and read the installed propagator attributes.
- [Follow segment](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSFollow.html): original helper offsets are defined in leader body axes; the new initial state specifies its rotating frame directly.

**Validation boundary:** reference geometry, finite-difference velocities, waypoint timing, burn merging and notebook syntax can be checked without Windows. The STK-dependent cells must be executed on the Windows STK 13 installation; no propagated results or convergence claims are pre-filled here. The embedded state precision and explicit Base-property choices limit numerical identity with the earlier scenario.
